# S11 · leakdepth 0.1.3 — imperfect-metadata benchmark, timed audits, exploratory label/overlap check (NB7)

Runs the protocol pre-registered as amendment **A14** (and the exploratory deviation **A13-D1**) before this notebook
was built. Nothing is tuned on these results.

| Cell | Item | What it does |
|---|---|---|
| 2 | M | imperfect-metadata benchmark on fresh seeds 300–319 (synthetic data, no download) |
| 3 | A | timed audits of the CIC-IDS2017 and UNSW-NB15 mirrors (each in its own process), attack host pairs, UNSW zero/missing shares before and after `blank_to_zero` |
| 4 | D1 | EXPLORATORY (deviation from A13): labels of the NB6 matched rows with the correct label columns, and exact-overlap checks without timestamps |
| 5 | Summary | compact JSON (paste it back) |

**Runtime.** Colab **High-RAM CPU**; roughly 1–1.5 h (downloads ≈ 1.6 GB). Drive is mounted once; no secrets or tokens
are requested. Outputs go to `MyDrive/S11-SecSplit/nb7/` as each job finishes. After a disconnect rerun cell 1, then
the interrupted cell; finished jobs are skipped.


In [ ]:
# 1. Setup: Drive (mounted once), embedded leakdepth 0.1.3 wheel (hash-checked), helpers. No secrets or tokens are used.
import base64, hashlib, json, os, subprocess, sys, time, glob, gc, re, importlib, zlib, urllib.request, resource
LOCAL = os.environ.get("NB7_LOCAL")                     # local smoke tests only; unset on Colab
SMOKE = bool(os.environ.get("NB7_SMOKE"))
if LOCAL:
    BASE = LOCAL
else:
    if not os.path.isdir("/content/drive/MyDrive"):     # mount once; re-running cells never re-prompts
        from google.colab import drive
        drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
OUT = f"{BASE}/S11-SecSplit/nb7"
os.makedirs(OUT, exist_ok=True)
WD = f"{LOCAL}/_wd" if LOCAL else "/content/nb7"
DATA = f"{LOCAL}/_data" if LOCAL else "/content/data"
os.makedirs(WD, exist_ok=True); os.makedirs(DATA, exist_ok=True)
WHEEL_SHA = "39c75233dd67aff40fdde09a97e47a461cd5d0180723d3b02113603960129a93"  # package files identical to the tested wheel bfd4c2a4fdbd…; author metadata set to the release author
WHEEL_B64 = "..."
w = f"{WD}/leakdepth-0.1.3-py3-none-any.whl"
open(w, "wb").write(base64.b64decode(WHEEL_B64))
assert hashlib.sha256(open(w, "rb").read()).hexdigest() == WHEEL_SHA, "0.1.3 wheel hash mismatch"
if not LOCAL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", w, "pyarrow"], check=True)
import numpy as np, pandas as pd, leakdepth as L
LL = importlib.import_module("leakdepth.lineage")
from leakdepth import FlowTable, audit_split, concat
from leakdepth.inject import COLUMNS, plant
from leakdepth.metrics import wilson
assert L.__version__ == "0.1.3", L.__version__
print("leakdepth", L.__version__, "| numpy", np.__version__, "| pandas", pd.__version__, "| CPUs", os.cpu_count(),
      "| RAM GB", round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1))
T0 = time.time()

def done(name):
    return os.path.exists(f"{OUT}/{name}")

def jdump(obj, name):
    tmp = f"{OUT}/{name}.tmp"
    json.dump(obj, open(tmp, "w"), default=lambda o: o.item() if hasattr(o, "item") else str(o))
    os.replace(tmp, f"{OUT}/{name}")

def jload(name):
    return json.load(open(f"{OUT}/{name}"))

FILES = {"cic2017": [("rdpahalavan/CIC-IDS2017", "main", "Network-Flows/CICIDS_Flow.parquet",
                      "e64f6adfbcb7f96b018cf36c731be64ef4de3353ccd7340b190590734a0dda35")],
         "unsw": [("Mouwiya/UNSW-NB15", "main", "data/train-00000-of-00002.parquet",
                   "2aada2a26d061111f4e8fb84e716f5f11264fee71abe04697d42cb89e488d047"),
                  ("Mouwiya/UNSW-NB15", "main", "data/train-00001-of-00002.parquet",
                   "7c6699ae967567977dee9e9193543b515255f4e1671ca79bf9ae65e7866ffef1")],
         "cic2018": [("c01dsnap/CIC-IDS2018", "236baa10495af451de38485123a90f60e7632bbd",
                      "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv",
                      "acff8bc61376ee031d80878ee6099e0b1a87a1bd711d8068298421418c9f8147"),
                     ("c01dsnap/CIC-IDS2018", "236baa10495af451de38485123a90f60e7632bbd",
                      "Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv",
                      "fa2947a8256d81ee9103ae16139d62d0e17aa23e696ee80d9e76fb51c01c9c4b")],
         "insdn": [("Sharukesh/INSDN", "f59330c8ff0f1b231b382e05fc41b1e8fb0df6b4", "Dataset.csv",
                    "a93d51e1c2debc7246728578f22d84a442f156628be64b8c9bae5ccc5dd88a42")]}

def fetch(repo, rev, f, sha):
    p = f"{DATA}/{repo.replace('/', '__')}/{f}"
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not (os.path.exists(p) and hashlib.sha256(open(p, "rb").read()).hexdigest() == sha):
        url = f"https://huggingface.co/datasets/{repo}/resolve/{rev}/{f}"
        with urllib.request.urlopen(url) as r, open(p + ".part", "wb") as fh:
            while True:
                b = r.read(1 << 22)
                if not b:
                    break
                fh.write(b)
        os.replace(p + ".part", p)
    h = hashlib.sha256(open(p, "rb").read()).hexdigest()
    assert h == sha, f"STOP: {repo}/{f} differs from the recorded file ({h})"
    return p

def paths(name):
    if LOCAL:
        return [f"{BASE}/smoke_{name}.parquet"]
    return [fetch(*x) for x in FILES[name]]

def load(name):                                         # identical reading to NB5/NB6
    parts = [pd.read_parquet(p) if p.endswith(".parquet") else pd.read_csv(p, low_memory=False) for p in paths(name)]
    return pd.concat(parts, ignore_index=True)


In [ ]:
# 2. M: imperfect-metadata benchmark (A14). Seeds 300-319, leaks dup/session/host/capture; resumable JSONL.
TYPE_FINDING = {"dup": "overlap-dup", "session": "overlap-session", "host": "overlap-hostpair", "capture": "overlap-capture"}
UNIT_BASE = {"dup": "dup", "session": "session", "host": "hostpair", "capture": "capture"}
DEFAULT_UNITS = ("row", "dup", "5tuple", "hostpair", "session:300", "capture")
SEEDS = list(range(300, 320)) if not SMOKE else [300, 301]
COND = [("M0", None)] + [("M1", None), ("M2", None), ("M3", None), ("M4", None)] + \
       [("M5", j) for j in (1, 60, 600)] + [("M6", m) for m in (0.5, 1.0)] + [("M7", g) for g in (30, 3000)] + [("M8", None)]
RHOS_M0 = (0.0, 0.001, 0.0025, 0.01, 0.05)
RHOS = (0.0, 0.01, 0.05)
ADDR = ("src_ip", "dst_ip")

def _addr(i, a, b):
    return f"{a}.{b}.{(i // 256) % 256}.{i % 256}"

def perturb(tr, te, cond, par, seed):
    rng = np.random.default_rng([seed, zlib.crc32(cond.encode()), int(1000 * (par or 0))])
    tr, te = tr.copy(), te.copy()
    if cond == "M1":
        u = pd.unique(pd.concat([tr[c] for c in ADDR] + [te[c] for c in ADDR]))
        m = dict(zip(u, (_addr(i, 172, 16 + i // 65536) for i in rng.permutation(len(u)))))
        for d in (tr, te):
            for c in ADDR:
                d[c] = d[c].map(m)
    elif cond == "M2":
        for d, a in ((tr, 172), (te, 192)):
            u = pd.unique(pd.concat([d[c] for c in ADDR]))
            m = dict(zip(u, (_addr(i, a, 16 + i // 65536) for i in rng.permutation(len(u)))))
            for c in ADDR:
                d[c] = d[c].map(m)
    elif cond == "M3":
        for d in (tr, te):
            for c in ADDR:
                d[c] = d[c].map(lambda a: f"203.0.113.{zlib.crc32(str(a).encode()) % 4 + 1}")
    elif cond == "M4":
        keys = pd.unique(pd.concat([d[c] + "|" + d["day"] for d in (tr, te) for c in ADDR]))
        repl = rng.random(len(keys)) < 0.5
        m, k = {}, 0
        for key_, r_ in zip(keys, repl):
            if r_:
                m[key_] = _addr(k, 198, 18 + k // 65536); k += 1
        for d in (tr, te):
            for c in ADDR:
                kk = d[c] + "|" + d["day"]
                d[c] = np.where(kk.isin(list(m)), kk.map(m), d[c])
    elif cond == "M5":
        for d in (tr, te):
            d["ts"] = d["ts"].astype(float) + rng.uniform(-par, par, len(d))
    elif cond == "M6":
        for d in (tr, te):
            ts = d["ts"].astype(float).to_numpy().copy()
            ts[rng.random(len(d)) < par] = np.nan
            d["ts"] = ts
    elif cond == "M8":
        out = []
        for d in (tr, te):
            rev = d.copy()
            rev["src_ip"], rev["dst_ip"] = d["dst_ip"].to_numpy(), d["src_ip"].to_numpy()
            rev["src_port"], rev["dst_port"] = d["dst_port"].to_numpy(), d["src_port"].to_numpy()
            out.append(pd.concat([d, rev], ignore_index=True))
        tr, te = out
    return tr, te

MF = f"{OUT}/M_imperfect.jsonl"
seen = set()
if os.path.exists(MF):
    seen = {(r["cond"], r["par"], r["leak"], r["rho"], r["seed"]) for r in map(json.loads, open(MF))}
t0 = time.time(); n_new = 0
for seed in SEEDS:
    for leak in ("dup", "session", "host", "capture"):
        for rho in RHOS_M0:
            todo = [(c, p) for c, p in COND if (c == "M0" or rho in RHOS) and (c, p, leak, rho, seed) not in seen]
            if not todo:
                continue
            pl = plant(leak, rho, seed)
            for cond, par in todo:
                tr, te = perturb(pl.train, pl.test, cond, par, seed)
                units = DEFAULT_UNITS if cond != "M7" else ("row", "dup", "5tuple", "hostpair", f"session:{par}", "capture")
                t, itr, ite = concat(FlowTable(tr, COLUMNS, "train"), FlowTable(te, COLUMNS, "test"))
                rep = audit_split(t, itr, ite, units=units, cross_capture=True)
                fired = sorted({f.id for f in rep.findings if f.severity in ("WARN", "ERROR")})
                ov = [r["overlap_share"] for r in rep.results["split_overlap"] if r["unit"].split(":")[0] == UNIT_BASE[leak]]
                row = {"cond": cond, "par": par, "leak": leak, "rho": rho, "seed": seed, "n_leaked": pl.n_leaked,
                       "detected": TYPE_FINDING[leak] in fired, "fired": fired, "overlap": ov[0] if ov else None}
                with open(MF, "a") as fh:
                    fh.write(json.dumps(row) + "\n")
                n_new += 1
    print("seed", seed, "done |", n_new, "new |", round(time.time() - t0), "s")

M = pd.DataFrame(list(map(json.loads, open(MF))))
M["par"] = M["par"].fillna("-").astype(str)
def wl(x):
    k, n = int(x.sum()), len(x); lo, hi = wilson(k, n); return f"{k}/{n} [{lo:.2f}, {hi:.2f}]"
cell = M.groupby(["cond", "par", "leak", "rho"])["detected"].agg(["mean", "size", wl]).reset_index()
cell.to_csv(f"{OUT}/M_cells.csv", index=False)
lab = []
for (c, p, lk), g in M.groupby(["cond", "par", "leak"]):
    fp = g[g.rho == 0]["detected"].mean()
    det = [g[g.rho == r]["detected"].mean() for r in (0.01, 0.05)]
    lab.append({"cond": c, "par": p, "leak": lk, "fp": round(fp, 3), "det_0.01": round(det[0], 3), "det_0.05": round(det[1], 3),
                **({f"det_{r}": round(g[g.rho == r]["detected"].mean(), 3) for r in (0.001, 0.0025)} if c == "M0" else {}),
                "label": "robust" if min(det) >= 0.95 and fp <= 1 / 20 else "degraded",
                "overlap_mean_0.05": round(float(g[g.rho == 0.05]["overlap"].mean()), 4)})
LAB = pd.DataFrame(lab)
LAB.to_csv(f"{OUT}/M_labels.csv", index=False)
with pd.option_context("display.width", 220, "display.max_rows", 200):
    print(LAB.to_string(index=False))


In [ ]:
# 3. A: timed audits (A14), each in its own process so that time and peak memory are its own.
CHILD = r'''
import json, os, resource, sys, time
import numpy as np, pandas as pd, leakdepth as L
from leakdepth import groups as G
job, out = sys.argv[1], sys.argv[2]
files = json.loads(sys.argv[3])
t0 = time.time()
df = pd.concat([pd.read_parquet(p) for p in files], ignore_index=True)
t1 = time.time()
res = {"job": job, "cpus": os.cpu_count(), "rows": len(df)}
if job == "cic":
    t = L.load(df, "cic-ids2017", name="CIC-IDS2017 (HF mirror)")
    del df
    t2 = time.time(); rep = L.audit(t); t3 = time.time()
    g = G.extract(t, "hostpair"); y = t.y.astype(bool); cls = t.cls
    att = pd.Series(g[y]).value_counts()
    res |= {"host_pairs": int(len(np.unique(g))), "host_pairs_with_attacks": int(len(att)),
            "attack_rows": int(y.sum()), "share_attacks_top1_pair": float(att.iloc[:1].sum() / y.sum()),
            "share_attacks_top5_pairs": float(att.iloc[:5].sum() / y.sum()),
            "host_pairs_per_class": {str(c): int(len(np.unique(g[cls == c]))) for c in sorted(set(cls[y]))}}
else:
    btz = job == "unsw_btz1"
    t = L.load(df, "unsw-nb15", name="UNSW-NB15 (HF mirror)", blank_to_zero=btz)
    del df
    t2 = time.time(); rep = L.audit(t); t3 = time.time()
    cap = t.meta("capture")
    per = {}
    for c in ("ct_flw_http_mthd", "is_ftp_login", "ct_ftp_cmd"):
        v = pd.to_numeric(t.df[c], errors="coerce")
        per[c] = {str(k): {"missing": float(v[cap == k].isna().mean()), "zero": float((v[cap == k] == 0).mean())}
                  for k in sorted(set(cap))}
    zs = {}
    for c in t.feature_names:
        v = pd.to_numeric(t.df[c], errors="coerce")
        z = [(v[cap == k] == 0).mean() for k in sorted(set(cap))]
        zs[c] = float(max(z) - min(z))
    res |= {"blank_to_zero": btz, "periods": per, "columns_zero_share_diff_ge_0.5": sorted(c for c, d in zs.items() if d >= 0.5)}
res |= {"load_s": round(t1 - t0, 1), "profile_s": round(t2 - t1, 1), "audit_s": round(t3 - t2, 1),
        "peak_rss_gb": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2),
        "findings": [(f.id, f.severity, f.unit) for f in rep.findings]}
json.dump(res, open(out, "w"))
'''
open(f"{WD}/audit_child.py", "w").write(CHILD)
JOBS = [("cic", "cic2017"), ("unsw_btz0", "unsw"), ("unsw_btz1", "unsw")]
for job, ds in JOBS:
    fn = f"A_{job}.json"
    if not done(fn):
        files = paths(ds)
        r = subprocess.run([sys.executable, f"{WD}/audit_child.py", job, f"{OUT}/{fn}", json.dumps(files)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(job, "FAILED:", r.stderr[-2000:]); continue
    a = jload(fn)
    print(job, {k: a[k] for k in a if k not in ("findings", "periods", "host_pairs_per_class")})
    if "periods" in a:
        print("  ", json.dumps(a["periods"]))


In [ ]:
# 4. D1 — EXPLORATORY (deviation A13-D1 from the pre-registered A13): labels with the correct label columns, and
#    exact-overlap checks without timestamps, for the NB5/NB6 directions N1-N3. Same matcher (A10, 20 bits, seed 0).
SEED, SAMPLE_N, TAU, N_BANDS, CHUNK, MARGIN, MAX_COST, MIN_ALIGNED = 0, 200_000, 0.8, 8, 200_000, 20.0, 0.1, 4
EXPECT = {"cic2017 in cic2018": 11855, "cic2018 in cic2017": 43517, "insdn in cic2017": 647,
          "cic2017 in insdn": 4232, "insdn in cic2018": 435, "cic2018 in insdn": 5606}
LABEL = {"cic2017": "attack_label", "cic2018": "Label", "insdn": "target"}    # corrected names (A13-D1)
PAIRS = [("N1", "cic2017", "cic2018"), ("N2", "insdn", "cic2017"), ("N3", "insdn", "cic2018")]

def key(name):
    return re.sub(r"[^a-z]", "", str(name).lower())
EXCL = {"timestamp", "label", "attacklabel", "target", "flowid", "sourceip", "destinationip", "srcip", "dstip",
        "sourceport", "srcport"}

def family(lbl):
    s = str(lbl).lower()
    if re.search(r"benign|normal", s): return "benign"
    if re.search(r"web|xss|sql", s): return "web"
    if "ddos" in s: return "ddos"
    if re.search(r"slowloris|slowhttp", s): return "dos-slow"
    if re.search(r"hulk|goldeneye|dos", s): return "dos-other"
    if re.search(r"ftp|ssh|patator|bfa|brute", s): return "brute"
    if re.search(r"portscan|probe|scan", s): return "scan"
    if "bot" in s: return "bot"
    return "other"

def fam(ds, v):                                         # InSDN mirror: binary target (1 = attack), no attack types
    if ds == "insdn":
        return "benign" if str(v).strip() in ("0", "0.0") else "attack"
    return family(v)

def top(s, k):
    vc = pd.Series(s).astype(str).value_counts(normalize=True)
    return {str(i): round(float(v), 4) for i, v in vc.head(k).items()}

def combined(df, cols, kinds, decs):
    c = LL.canonicalise(df, cols, kinds, decs)
    return LL._combine(c.col_hashes(), np.arange(c.k))

def shared_exact(a, b):
    na = {LL.norm_name(c): c for c in a.columns if key(c) not in EXCL}
    nb = {LL.norm_name(c): c for c in b.columns if key(c) not in EXCL}
    sh = sorted(set(na) & set(nb) - {"timestamp", "label"})
    if len(sh) < 4:
        return {"shared": sh, "checked": False}
    def h(df, cols):
        parts = []
        for c in cols:
            v = pd.to_numeric(df[c], errors="coerce")
            parts.append(np.where(v.notna(), v.round(9).astype(str), df[c].astype(str).str.strip().str.lower()))
        return pd.util.hash_pandas_object(pd.DataFrame(dict(enumerate(parts))), index=False).to_numpy()
    ha, hb = h(a, [na[s] for s in sh]), h(b, [nb[s] for s in sh])
    return {"shared": sh, "checked": True, "a_in_b": float(np.isin(ha, hb).mean()), "b_in_a": float(np.isin(hb, ha).mean())}

CACHE = {}
def get(n):
    if n not in CACHE:
        CACHE[n] = load(n)
        if SMOKE:
            CACHE[n] = CACHE[n].sample(n=min(20000, len(CACHE[n])), random_state=0).reset_index(drop=True)
    return CACHE[n]

for pid, an, bn in PAIRS:
    fn = f"D1_{pid}.json"
    if done(fn):
        continue
    a, b = get(an), get(bn)
    out = {"pair": pid, "shared_name_exact_overlap_no_time": shared_exact(a, b), "directions": {}}
    pa, pb = LL.profile_columns(a, SAMPLE_N, SEED), LL.profile_columns(b, SAMPLE_N, SEED)
    inf = [x for x in LL.align_columns(pa, pb, MAX_COST) if pa[x[0]].kind != "const"]
    for nx, ny, x, y, px, py, al in ((an, bn, a, b, pa, pb, inf), (bn, an, b, a, pb, pa, [(q, p, c, h) for p, q, c, h in inf])):
        sx = (x.sample(n=min(SAMPLE_N, len(x)), random_state=SEED) if len(x) > SAMPLE_N else x).reset_index(drop=True)
        res, used = LL._containment(sx, y, px, py, al, TAU, N_BANDS, SEED, CHUNK, min_aligned=MIN_ALIGNED, info_margin=MARGIN)
        d = f"{nx} in {ny}"; mi = np.flatnonzero(res["matched"]); part = res["partner"][mi]
        r = {"matched": int(len(mi)), "expected_nb5": EXPECT.get(d), "reproduced": int(len(mi)) == EXPECT.get(d)}
        la, lb = LABEL[nx], LABEL[ny]
        if la in sx.columns and lb in y.columns and len(mi):
            A = sx[la].iloc[mi].astype(str).str.strip().to_numpy(); B = y[lb].iloc[part].astype(str).str.strip().to_numpy()
            fa = np.array([fam(nx, v) for v in A]); fb = np.array([fam(ny, v) for v in B])
            fs = np.array([fam(nx, v) for v in sx[la].astype(str).str.strip()])
            coarse = "insdn" in (nx, ny)                # InSDN has no attack types: compare attack vs benign only
            r |= {"labels_a_top10": top(A, 10), "labels_b_top10": top(B, 10), "label_pairs_top10": top(pd.Series(A) + " -> " + pd.Series(B), 10),
                  "nonbenign_a": round(float(np.mean(fa != "benign")), 4), "nonbenign_b": round(float(np.mean(fb != "benign")), 4),
                  "same_family_nonbenign": round(float(np.mean(((fa == fb) | coarse) & (fa != "benign") & (fb != "benign"))), 4),
                  "family_comparison": "attack vs benign only (InSDN binary)" if coarse else "attack family",
                  "family_pairs": top(pd.Series(fa) + " -> " + pd.Series(fb), 10),
                  "baseline_nonbenign_a_sample": round(float(np.mean(fs != "benign")), 4)}
            h1 = r["nonbenign_a"] >= 0.5
            r["H1_exploratory"] = {"attack_dominated": h1, "same_family_attack_flows": bool(h1 and r["same_family_nonbenign"] >= 0.5)}
        else:
            r["labels"] = f"missing: {la} in A: {la in sx.columns}; {lb} in B: {lb in y.columns}"
        keep = [u for u in used if key(u[0]) not in EXCL and key(u[1]) not in EXCL]
        if len(keep) >= MIN_ALIGNED:
            a_cols = [u[0] for u in keep]; b_cols = [u[1] for u in keep]
            kinds = ["num" if px[c].kind == "num" else "str" for c in a_cols]
            decs = []
            for ca, cb, k in zip(a_cols, b_cols, kinds):
                dd = [q for q in (px[ca].decimals, py[cb].decimals) if q is not None] if k == "num" else []
                decs.append(min(dd) if dd else None)
            ha = combined(sx, a_cols, kinds, decs)
            hb = np.concatenate([combined(y.iloc[i:i + CHUNK], b_cols, kinds, decs) for i in range(0, len(y), CHUNK)])
            inb = np.isin(ha, hb)
            r["aligned_exact_overlap_no_time"] = {"columns": len(keep), "share_of_sample": round(float(inb.mean()), 5),
                                                  "share_of_matched": round(float(inb[mi].mean()), 4) if len(mi) else None}
        out["directions"][d] = r
        print(pid, d, {k: v for k, v in r.items() if k in ("matched", "reproduced", "nonbenign_a", "same_family_nonbenign", "aligned_exact_overlap_no_time")})
    print(pid, "shared-name exact overlap without time/label/ids:", {k: v for k, v in out["shared_name_exact_overlap_no_time"].items() if k != "shared"})
    jdump(out, fn)
    for nm in list(CACHE):
        if all(nm not in (p[1], p[2]) for p in PAIRS[PAIRS.index((pid, an, bn)) + 1:]):
            del CACHE[nm]
    gc.collect()


In [ ]:
# 5. Compact summary (paste this output back); also nb7/S11_NB7_summary.json
S = {"leakdepth": L.__version__}
if os.path.exists(f"{OUT}/M_labels.csv"):
    S["M"] = pd.read_csv(f"{OUT}/M_labels.csv").to_dict("records")
S["A"] = {j: {k: v for k, v in jload(f"A_{j}.json").items() if k != "findings"} | {"n_findings": len(jload(f"A_{j}.json")["findings"])}
          for j in ("cic", "unsw_btz0", "unsw_btz1") if done(f"A_{j}.json")}
S["D1_exploratory"] = {p: jload(f"D1_{p}.json") for p in ("N1", "N2", "N3") if done(f"D1_{p}.json")}
for p in S["D1_exploratory"].values():
    p["shared_name_exact_overlap_no_time"].pop("shared", None)
S["runtime_min_this_session"] = round((time.time() - T0) / 60, 1)
jdump(S, "S11_NB7_summary.json")
print(json.dumps(S, separators=(",", ":"), default=str))
